[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/notebooks/19_RLM_ejercicio_tiktok.ipynb)

# Ejercicio integrador: ¿qué provoca la compra impulsiva en TikTok Live Shopping?


## El caso

El *live shopping* (transmisiones en vivo donde un anfitrión muestra productos y los espectadores compran sin salir de la aplicación) es uno de los formatos de comercio electrónico de más rápido crecimiento. A diferencia de una tienda en línea tradicional, la venta ocurre en tiempo real, con un vendedor carismático, un cronómetro que corre y una audiencia que reacciona en el chat. El resultado es que una parte importante de las compras **no son planeadas**.

Para una empresa que invierte en este canal, la pregunta operativa es muy concreta: *¿en qué debo invertir para que la gente compre?* Hay tres candidatos naturales:

1. **Contratar mejores anfitriones** (más carismáticos, que conozcan el producto, que respondan rápido).
2. **Producir transmisiones más emocionantes** (ambiente, ritmo, entusiasmo, sensación de evento).
3. **Vender productos de mejor calidad** (durables, confiables, de marcas en las que se confía).

Los tres suenan razonables y los tres cuestan dinero. Este ejercicio usa datos reales de una encuesta a 136 usuarios de TikTok para estimar cuál de los tres realmente predice la compra impulsiva.

**Lo que vas a practicar:**
- Construir constructos (índices) a partir de ítems de una encuesta y verificar su confiabilidad.
- Estimar e interpretar un modelo de regresión lineal múltiple.
- Entender por qué una variable puede ser significativa por sí sola y dejar de serlo dentro del modelo.
- Evaluar los supuestos del modelo y decidir qué hacer cuando alguno no se cumple.
- Traducir los coeficientes en una recomendación de negocio.

## Los datos

Los datos provienen de un estudio cuantitativo sobre compra impulsiva en TikTok Live. Cada persona respondió 53 afirmaciones en escala Likert de 1 (*totalmente en desacuerdo*) a 5 (*totalmente de acuerdo*), agrupadas en bloques.

**Diccionario de variables**

| Ítems | Constructo | Qué mide | Sub-dimensiones |
|---|---|---|---|
| A1–A10 | **Desempeño del anfitrión** | Qué tan bien conduce la transmisión quien vende | claridad (A1–A2), capacidad de respuesta (A3–A4), interactividad (A5–A6), conocimiento del producto (A7–A8), carisma (A9–A10) |
| C1–C11 | **Euforia emocional** | La activación emocional que produce ver la transmisión | júbilo (C1–C3), satisfacción emocional (C4–C5), emoción imaginada (C6–C7), entusiasmo por el ambiente del live (C8–C9), detonante emocional (C10–C11) |
| D1–D10 | **Valor de calidad** | La calidad percibida del producto ofrecido | durabilidad (D1–D2), confiabilidad del desempeño (D3–D4), apariencia (D5–D6), satisfacción del consumidor (D7–D8), confianza en la marca (D9–D10) |
| E1–E11 | **Compra impulsiva** *(variable dependiente)* | Tendencia a comprar sin planearlo durante la transmisión | compra no planeada (E1–E2), falta de control (E3–E5), reacción a estímulos visuales (E6–E7), decisión apresurada (E8–E9), gratificación instantánea (E10–E11) |

> **Nota**: el archivo incluye además un bloque B1–B11 que no forma parte del modelo publicado del estudio; lo dejaremos fuera del análisis principal, aunque al final se propone como reto explorarlo.

*¿Por qué tantos ítems para medir una sola cosa?* Porque conceptos como "euforia" o "impulsividad" no se pueden medir con una sola pregunta de forma confiable. Una sola pregunta captura mucho ruido: la persona pudo haberla entendido mal, o su respuesta pudo depender del estado de ánimo de ese día. Al preguntar lo mismo de varias maneras y promediar, el ruido tiende a cancelarse y queda la señal. A ese promedio se le llama **constructo**, **índice** o **variable latente**.

## Preparación de los datos

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
# Cargar los datos
url = 'https://github.com/adan-rs/amd/raw/main/data/tiktok_live.xlsx'
df = pd.read_excel(url)

# Si trabajas con el archivo local en lugar del repositorio, usa:
# df = pd.read_excel('tiktok_live.xlsx')

In [ ]:
# Revisar las variables y el número de observaciones
df.info()

In [ ]:
# Revisar las primeras filas
df.head(3)

In [ ]:
# Eliminar los casos con datos faltantes
df = df.dropna().reset_index(drop=True)
print(f'Observaciones para el análisis: {len(df)}')

## Paso 1. De ítems a constructos

Un modelo de regresión con 44 ítems individuales como variables independientes sería imposible de interpretar y estaría plagado de multicolinealidad (los ítems de un mismo bloque miden casi lo mismo). Por eso el primer paso es **reducir cada bloque de ítems a una sola variable**, calculando el promedio de sus ítems para cada persona.

Esta parte del código ya está resuelta: el foco del ejercicio está en la interpretación del modelo, no en la construcción de las variables.

In [ ]:
# Definir qué ítems componen cada constructo
constructos = {
    'anfitrion': [f'A{i}' for i in range(1, 11)],   # A1 a A10
    'euforia':   [f'C{i}' for i in range(1, 12)],   # C1 a C11
    'calidad':   [f'D{i}' for i in range(1, 11)],   # D1 a D10
    'impulsiva': [f'E{i}' for i in range(1, 12)]    # E1 a E11
}

# Calcular el promedio de los ítems de cada bloque, fila por fila
for nombre, items in constructos.items():
    df[nombre] = df[items].mean(axis=1)

df[['anfitrion', 'euforia', 'calidad', 'impulsiva']].head()

### Confiabilidad de los constructos

Antes de usar un índice hay que verificar que sus ítems efectivamente midan lo mismo. La medida estándar es el **alfa de Cronbach**, que toma valores entre 0 y 1:

- α ≥ 0.70 → confiabilidad aceptable
- α ≥ 0.80 → buena
- α ≥ 0.90 → excelente (aunque valores muy altos, arriba de 0.95, pueden indicar ítems redundantes)

Si un constructo tuviera un alfa bajo, promediar sus ítems no tendría sentido: estaríamos sumando peras con manzanas.

In [ ]:
def alfa_cronbach(items_df):
    """Calcula el alfa de Cronbach de un conjunto de ítems."""
    k = items_df.shape[1]
    var_items = items_df.var(ddof=1).sum()
    var_total = items_df.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - var_items / var_total)


confiabilidad = pd.DataFrame({
    'Constructo': list(constructos.keys()),
    'N de ítems': [len(v) for v in constructos.values()],
    'Alfa de Cronbach': [alfa_cronbach(df[v]) for v in constructos.values()]
})
confiabilidad.round(3)

## Paso 2. Exploración descriptiva

Antes de estimar cualquier modelo, conviene mirar los datos.

In [ ]:
# Estadística descriptiva de los constructos
var_modelo = ['anfitrion', 'euforia', 'calidad', 'impulsiva']
df[var_modelo].describe().T.round(3)

In [ ]:
# Matriz de correlaciones
df[var_modelo].corr().round(3)

In [ ]:
# Relaciones entre variables
sns.pairplot(df[var_modelo], corner=True, kind='reg', markers='+', height=1.6);

> **Pregunta guía 1.** Antes de correr el modelo, escribe tu predicción: viendo únicamente la matriz de correlaciones, ¿cuáles de las tres variables independientes esperas que resulten significativas para explicar la compra impulsiva? Anótalo ahora; lo vas a comparar con el resultado real más adelante.

## Paso 3. Estimación del modelo

El modelo a estimar es:

$$
\text{compra impulsiva} = \beta_0 + \beta_1 \text{anfitrión} + \beta_2 \text{euforia} + \beta_3 \text{calidad} + \varepsilon
$$

In [ ]:
X = df[['anfitrion', 'euforia', 'calidad']]
X = sm.add_constant(X)
y = df['impulsiva']

modelo = sm.OLS(y, X).fit()
modelo.summary()

> **Pregunta guía 2.** Con base en la salida anterior, responde:
> 1. ¿El modelo es significativo en su conjunto? ¿Con qué estadístico y qué valor p lo determinas?
> 2. ¿Qué porcentaje de la variabilidad de la compra impulsiva explica el modelo?
> 3. ¿Cuáles variables independientes resultaron estadísticamente significativas y cuáles no?
> 4. Interpreta el coeficiente de `euforia` en una frase, en términos del negocio (recuerda que ambas variables están en escala 1 a 5).
> 5. ¿Coincidió el resultado con la predicción que anotaste en la pregunta guía 1?

## Paso 4. La sorpresa: regresiones simples vs. modelo múltiple

El resultado anterior suele sorprender. Vamos a estimar ahora **tres regresiones simples**, una por cada variable independiente por separado, y a comparar los coeficientes con los del modelo múltiple.

In [ ]:
filas = []
for v in ['anfitrion', 'euforia', 'calidad']:
    simple = sm.OLS(y, sm.add_constant(df[[v]])).fit()
    filas.append({
        'Variable': v,
        'Coef. modelo simple': simple.params[v],
        'p simple': simple.pvalues[v],
        'R2 simple': simple.rsquared,
        'Coef. modelo múltiple': modelo.params[v],
        'p múltiple': modelo.pvalues[v]
    })

pd.DataFrame(filas).round(3)

> **Pregunta guía 3.** Las tres variables son significativas cuando se analizan por separado, pero dos de ellas dejan de serlo al entrar juntas al modelo. El coeficiente del anfitrión incluso cambia de signo.
> 1. ¿Qué significa exactamente la frase "manteniendo todo lo demás constante" a la luz de este resultado?
> 2. Si un directivo te dijera *"el estudio muestra que un mejor anfitrión aumenta la compra impulsiva"* citando la regresión simple, ¿qué le responderías?

## Paso 5. ¿Es multicolinealidad?

La primera hipótesis ante un resultado así suele ser la multicolinealidad: variables independientes tan correlacionadas entre sí que el modelo no puede separar sus efectos. Vamos a verificarlo con el **Factor de Inflación de la Varianza (VIF)**, donde valores mayores a 10 indicarían un problema grave.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data.round(2)

Los VIF de las tres variables independientes están muy por debajo del umbral de alarma. **La multicolinealidad no es la explicación.**

Lo que ocurre es distinto: la calidad del producto y el desempeño del anfitrión sí influyen en la compra impulsiva, pero **su efecto pasa a través de la euforia**. Un producto que se percibe de calidad y un anfitrión carismático generan activación emocional, y es esa activación la que dispara la compra no planeada. Cuando la euforia ya está en el modelo, el "camino" por el que viajaba el efecto de las otras dos variables ya está ocupado. A esto se le llama **mediación**.

Podemos ponerlo a prueba estimando el modelo del mediador: si la calidad predice la euforia, y la euforia predice la compra, entonces el efecto indirecto (el producto de ambos coeficientes) debería reproducir aproximadamente el efecto total que veíamos en la regresión simple.

In [ ]:
# Modelo del mediador: ¿qué explica la euforia?
med = sm.OLS(df['euforia'], sm.add_constant(df[['calidad']])).fit()
print(med.summary().tables[1])

efecto_indirecto = med.params['calidad'] * modelo.params['euforia']
efecto_total = sm.OLS(y, sm.add_constant(df[['calidad']])).fit().params['calidad']

print(f'\nEfecto indirecto de calidad (vía euforia): {efecto_indirecto:.3f}')
print(f'Efecto total de calidad (regresión simple): {efecto_total:.3f}')
print(f'Proporción del efecto que viaja por la euforia: {efecto_indirecto/efecto_total:.1%}')

> **Pregunta guía 4.** Prácticamente todo el efecto de la calidad sobre la compra impulsiva viaja a través de la euforia.
> 1. ¿Es correcto concluir entonces que "la calidad del producto no importa"? Justifica.
> 2. Repite el mismo cálculo para `anfitrion` y comenta el resultado.

## Paso 6. Evaluación de supuestos

Ningún resultado de regresión debe reportarse sin revisar los supuestos del modelo.

### Homocedasticidad

In [ ]:
residuales = modelo.resid
y_hat = modelo.fittedvalues

plt.figure(figsize=(6, 3))
plt.scatter(y_hat, residuales, marker='.', color='b')
plt.axhline(0, color='gray', linewidth=0.8)
plt.xlabel('y hat')
plt.ylabel('residuales');

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

lm, lm_p_value, fvalue, f_p_value = het_breuschpagan(residuales, X)
print("Estadístico LM:", round(lm, 4))
print("Valor p del estadístico LM:", round(lm_p_value, 4))

### Normalidad de los residuales

In [ ]:
plt.figure(figsize=(4, 3))
sns.histplot(residuales, color='b', kde=True);

In [ ]:
from scipy.stats import shapiro

stat, p_value = shapiro(residuales)
print("Estadístico de prueba:", round(stat, 4))
print("Valor p:", round(p_value, 6))

alpha = 0.05
if p_value < alpha:
    print("Se rechaza la hipótesis nula de normalidad en los datos")
else:
    print("No se puede rechazar la hipótesis nula de normalidad en los datos")

> **Pregunta guía 5.** Los dos supuestos anteriores presentan problemas en este modelo.
> 1. ¿Qué consecuencia tiene la heterocedasticidad sobre los errores estándar y, por lo tanto, sobre los valores p que ya interpretaste?
> 2. La no normalidad de los residuales, ¿es igual de preocupante con n = 135 que con una muestra de 20 casos? ¿Por qué?
> 3. Revisa el histograma: el sesgo es negativo. ¿Qué te dice eso sobre el tipo de personas a las que el modelo les falla más?

### Una solución: errores estándar robustos

Cuando se detecta heterocedasticidad, una alternativa práctica (más simple que transformar variables o usar WLS) es estimar **errores estándar robustos a heterocedasticidad**, conocidos como errores de White o HC. Los coeficientes no cambian; lo que cambia es su error estándar y, por lo tanto, los valores p y los intervalos de confianza.

In [ ]:
modelo_robusto = sm.OLS(y, X).fit(cov_type='HC3')
print(modelo_robusto.summary().tables[1])

> **Pregunta guía 6.** Compara esta salida con la del modelo original. ¿Cambian las conclusiones sustantivas del estudio? ¿Qué te dice eso sobre la solidez del hallazgo?

## Paso 7. Comparar la magnitud de los efectos

Los coeficientes de este modelo son comparables entre sí porque las tres variables están en la misma escala (1 a 5). Cuando no es así, se recurre a los **coeficientes estandarizados** (beta), que expresan el cambio en desviaciones estándar de *y* ante un cambio de una desviación estándar en *x*.

In [ ]:
z = (df[var_modelo] - df[var_modelo].mean()) / df[var_modelo].std()
modelo_z = sm.OLS(z['impulsiva'], sm.add_constant(z[['anfitrion', 'euforia', 'calidad']])).fit()

pd.DataFrame({'Beta estandarizado': modelo_z.params.drop('const')}).round(3)

## Ejemplo de reporte de resultados

>"Se estimó un modelo de regresión lineal múltiple para explicar la compra impulsiva en transmisiones de TikTok Live (n = 135) a partir de tres constructos: desempeño del anfitrión, euforia emocional y valor de calidad percibido. Los cuatro constructos se calcularon como el promedio de sus ítems y mostraron confiabilidad de buena a excelente (alfa de Cronbach entre 0.881 y 0.934). El modelo resultó significativo (F(3, 131) = 65.65, p < 0.001) y explicó el 60.1% de la varianza de la compra impulsiva (R² = 0.601; R² ajustada = 0.591). Únicamente la euforia emocional resultó estadísticamente significativa (β = 1.044, p < 0.001; beta estandarizado = 0.773), mientras que el desempeño del anfitrión (β = −0.108, p = 0.371) y el valor de calidad (β = 0.071, p = 0.576) no lo fueron, pese a que ambas variables sí eran significativas al analizarse de forma aislada. Los factores de inflación de la varianza (entre 1.85 y 2.59) descartan la multicolinealidad como explicación; el análisis del modelo mediador indica que el 99% del efecto total de la calidad percibida sobre la compra impulsiva se transmite a través de la euforia emocional. La prueba de Breusch-Pagan detectó heterocedasticidad (p = 0.005), por lo que los contrastes se reestimaron con errores estándar robustos HC3, sin que cambiaran las conclusiones sustantivas."

Observa la estructura: primero la construcción y confiabilidad de las variables, luego el ajuste global, después los coeficientes individuales, en seguida los diagnósticos y las medidas correctivas. Los números se reportan con el estadístico, los grados de libertad y el valor p.

## Ejercicio a entregar

Con base en todo el análisis anterior, elabora un reporte breve (máximo dos cuartillas) que incluya:

1. **Descripción de las variables**: qué constructos se midieron, cómo se construyeron y qué confiabilidad tienen.
2. **Resultados del modelo**: ajuste global, coeficientes y su significancia, redactados en prosa como en el ejemplo anterior.
3. **Diagnóstico de supuestos**: qué se evaluó, qué se encontró y qué se hizo al respecto.
4. **Discusión del hallazgo central**: explica, para alguien que no sabe estadística, por qué el anfitrión y la calidad dejan de importar en el modelo múltiple y por qué eso *no* significa que sean irrelevantes para el negocio.
5. **Recomendación de negocio**: una empresa tiene un presupuesto limitado y debe elegir entre (a) contratar anfitriones más carismáticos, (b) invertir en producción para hacer transmisiones más emocionantes, o (c) mejorar la calidad de los productos que ofrece. ¿Qué le recomiendas con base en la evidencia? Señala al menos una limitación de tu recomendación.
6. **Limitaciones del estudio**: considera el tamaño de muestra, el hecho de que todas las variables se midieron con el mismo cuestionario al mismo tiempo (¿puedes afirmar causalidad?) y el rango restringido de las respuestas.

## Reto opcional *(perfil avanzado)*

*Este reto no es evaluado; está pensado para quienes deseen profundizar.*

**A. Abrir la caja negra de la euforia.** La euforia es un constructo de 11 ítems agrupados en cinco sub-dimensiones (júbilo, satisfacción emocional, emoción imaginada, ambiente del live y detonante emocional). Construye las cinco sub-dimensiones por separado y estima un modelo con ellas. ¿Cuáles son las que realmente empujan la compra impulsiva? ¿Qué implicación práctica tendría para el diseño de una transmisión?

**B. El bloque olvidado.** El archivo contiene un bloque B1–B11 que quedó fuera del modelo publicado. Constrúyelo como constructo, revisa su alfa de Cronbach y sus correlaciones con el resto. ¿Aporta algo al modelo? Usa AIC y BIC para decidir, no solo la R².

**C. Selección de modelos.** Aplica eliminación hacia atrás y regresión Lasso (notebook 17) sobre un modelo que incluya las cinco sub-dimensiones de euforia más el anfitrión y la calidad. ¿Coinciden ambos métodos en las variables que retienen?

In [ ]:
# Pistas para el reto A (completa el código)
# sub_euforia = {
#     'jubilo':    ['C1', 'C2', 'C3'],
#     'satisfacc': ['C4', 'C5'],
#     'imaginada': ['C6', 'C7'],
#     'ambiente':  ['C8', 'C9'],
#     'detonante': ['C10', 'C11']
# }
# for nombre, items in sub_euforia.items():
#     df[nombre] = df[items].mean(axis=1)
#
# X_sub = sm.add_constant(df[list(sub_euforia.keys())])
# sm.OLS(y, X_sub).fit().summary()

**Fuente de los datos**: Prameswari, A. *Research Dataset – Quantitative* (encuesta a usuarios de TikTok Live, n = 136). El modelo original fue estimado con PLS-SEM; en este ejercicio se replica con regresión por mínimos cuadrados ordinarios sobre índices promediados, lo que constituye una simplificación deliberada con fines didácticos.